<a href="https://www.kaggle.com/code/zn1350/chess-game-training-generate-data-cpu?scriptVersionId=335282171" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

`gen_data` 代码**没有使用 GPU**。它通过 `multiprocessing.Pool` 使用多个 CPU 核心并行生成对局，AI 引擎内部也是纯手工评估，完全不依赖 CUDA 或神经网络。

但是，在当前的约束下**无法将它修改为有效的 GPU 加速**，原因如下：

1. **评估函数是纯手工特征**  
   生成数据时强制 `disable_nn=True`，使用的是基于棋盘规则的手工评估。这种评估包含大量分支判断（如判断将军、缺士象、子力位置分等），无法被 GPU 高效并行，强行搬到 GPU 上反而会因为频繁的 kernel 调用和同步造成更严重的延迟。

2. **Alpha‑Beta 搜索是高度串行的**  
   每一步走法选择都需要递归地展开博弈树，并逐节点调用评估函数。GPU 擅长的是大规模数据并行计算（如矩阵乘法），而树搜索这种充满条件分支和循环依赖的任务在 GPU 上极难高效实现。

3. **多进程 CPU 已经是最合适的加速方式**  
   你的代码已经通过 `mp.Pool` 将总对局数分摊到多个进程上，在 8 核 CPU 上能将总时间缩短到接近 1/6（取决于 I/O 开销）。在“2 张 T4、总耗时 < 5 小时”的限制下，训练阶段（步骤 4）已经占了 GPU，数据生成放在 CPU 上完全不冲突，是最划算的资源分配方案。

### 如果一定要“看起来用了 GPU”
唯一有意义的改动是将**完全独立的对局**作为并行任务，用 CUDA 流或者 Numba CUDA 来并行多个对局？由于每个对局内部的搜索都是 CPU 密集型，GPU 只会拖慢速度，所以不建议。

**结论**：当前 `gen_data` 无法、也不应该被改为 GPU 加速，CPU 多进程方案已经是最优解。建议保持原样，把 GPU 留给 `train_micro.py`。

In [1]:
import os

dir_path = "/kaggle/input/models/zn1350/chess-game/pytorch/default/6"

# 只列当前目录下的文件（不含子目录）
print("【当前目录文件】")
for f in os.listdir(dir_path):
    f_path = os.path.join(dir_path, f)
    if os.path.isfile(f_path):
        print(f)

# 如果目录下还有子目录，想递归列出所有文件，用下面这段
print("\n【递归所有文件】")
for root, dirs, files in os.walk(dir_path):
    for f in files:
        print(os.path.join(root, f))

【当前目录文件】
gen_data.py
nn_evaluator.py
ui.py
main.py
train_micro.py
chess_game.py
config.py
assessment.py
model_improved.pth
debug_gesture.py
vision.py
requirements.txt
features.py
gen_data.ipynb
gesture_rl.py
ai_engine.py
debug_gesture_cli.py

【递归所有文件】
/kaggle/input/models/zn1350/chess-game/pytorch/default/6/gen_data.py
/kaggle/input/models/zn1350/chess-game/pytorch/default/6/nn_evaluator.py
/kaggle/input/models/zn1350/chess-game/pytorch/default/6/ui.py
/kaggle/input/models/zn1350/chess-game/pytorch/default/6/main.py
/kaggle/input/models/zn1350/chess-game/pytorch/default/6/train_micro.py
/kaggle/input/models/zn1350/chess-game/pytorch/default/6/chess_game.py
/kaggle/input/models/zn1350/chess-game/pytorch/default/6/config.py
/kaggle/input/models/zn1350/chess-game/pytorch/default/6/assessment.py
/kaggle/input/models/zn1350/chess-game/pytorch/default/6/model_improved.pth
/kaggle/input/models/zn1350/chess-game/pytorch/default/6/debug_gesture.py
/kaggle/input/models/zn1350/chess-game/pytorch/

In [2]:
#cell1 环境配置与依赖导入
import sys
import multiprocessing as mp
import time
import numpy as np

# ===================== 可修改配置（已适配10分钟跑完） =====================
class GenConfig:
    # 依赖脚本所在路径：替换为你上传的依赖数据集名称
    dependency_path = "/kaggle/input/models/zn1350/chess-game/pytorch/default/6"
    # 输出文件路径（Kaggle仅working目录可写，训练脚本直接读取这个文件）
    out_path = "/kaggle/working/selfplay_data.npz"
    
    # 生成参数（已调优适配10分钟耗时）
    games = 1500          # 总对局数
    workers = 6           # 并行进程数（Kaggle 8核CPU建议设6）
    depth = 2             # 搜索深度
    max_moves = 120       # 单局最大步数
    progress_every = 100  # 每完成多少局打印一次进度

# 添加依赖脚本到系统路径
sys.path.append(GenConfig.dependency_path)
from ai_engine import AIEngine
from chess_game import ChessGame, RED, BLACK
from features import extract_features

# 打印环境信息
print(f"CPU核心数: {mp.cpu_count()}, 启用进程数: {GenConfig.workers}")
print(f"配置参数：对局数={GenConfig.games}, 搜索深度={GenConfig.depth}, 单局上限={GenConfig.max_moves}步")

CPU核心数: 4, 启用进程数: 6
配置参数：对局数=1500, 搜索深度=2, 单局上限=120步


In [3]:
#cell2 单局对弈核心函数
def _play_one_game(args: tuple) -> list:
    """子进程内完成一局对弈，返回该局所有带标签的局面数据"""
    _game_idx, depth, max_moves = args
    engine = AIEngine(disable_nn=True)
    game = ChessGame()

    steps = []
    ply = 0
    while not game.game_over and ply < max_moves:
        color = game.current_turn
        feat = extract_features(game)
        board_snapshot = np.array(game.board, dtype=np.int8)
        best_move, move_scores = engine.get_move_with_scores(game, depth=depth)
        if best_move is None:
            break
        moves = list(move_scores.keys())
        scores = [move_scores[m] for m in moves]
        steps.append((feat, board_snapshot, scores, moves, color))
        game.make_move(*best_move)
        ply += 1

    # 生成最终结果标签（红方视角）
    if game.game_over and game.winner is not None:
        result = game.winner
    else:
        result = 0

    labeled = []
    for feat, board_snapshot, scores, moves, color in steps:
        z = result if color == RED else -result
        labeled.append((feat, board_snapshot, scores, moves, z))
    return labeled

In [4]:
#cell3 并行生成主逻辑
print("=== 开始生成自我对弈数据 ===")
t_start = time.time()

# 生成任务列表
tasks = [(i, GenConfig.depth, GenConfig.max_moves) for i in range(GenConfig.games)]

# 预分配存储列表
all_features = []
all_boards = []
all_move_scores = []
all_moves_order = []
all_z = []

completed = 0
# 多进程并行生成
with mp.Pool(processes=GenConfig.workers) as pool:
    for game_steps in pool.imap_unordered(_play_one_game, tasks):
        # 批量追加数据
        for feat, board_snapshot, scores, moves, z in game_steps:
            all_features.append(feat)
            all_boards.append(board_snapshot)
            all_move_scores.append(scores)
            all_moves_order.append(moves)
            all_z.append(z)
        completed += 1
        
        # 进度打印
        if completed % GenConfig.progress_every == 0 or completed == GenConfig.games:
            elapsed = time.time() - t_start
            rate = completed / elapsed if elapsed > 0 else 0.0
            remaining = (GenConfig.games - completed) / rate if rate > 0 else 0.0
            print(
                f"[{completed}/{GenConfig.games}] 累计局面数={len(all_features)} "
                f"耗时={elapsed:.1f}s 速度={rate:.2f}局/秒 预估剩余={remaining/60:.1f}分钟"
            )

# 转换为numpy数组并保存
print("=== 数据整理中，正在保存文件 ===")
features_arr = np.stack(all_features, axis=0).astype(np.float32)
boards_arr = np.stack(all_boards, axis=0).astype(np.int8)
z_arr = np.array(all_z, dtype=np.float32)
move_scores_arr = np.array(all_move_scores, dtype=object)
moves_order_arr = np.array(all_moves_order, dtype=object)

np.savez(
    GenConfig.out_path,
    features=features_arr,
    boards=boards_arr,
    move_scores=move_scores_arr,
    z_values=z_arr,
    moves_order=moves_order_arr,
)

total_time = time.time() - t_start
print(f"\n✅ 生成完成！")
print(f"总对局数：{GenConfig.games} 局，总局面数：{len(all_features)} 个")
print(f"总耗时：{total_time/60:.1f} 分钟")
print(f"文件已保存至：{GenConfig.out_path}")

=== 开始生成自我对弈数据 ===
[100/1500] 累计局面数=12000 耗时=2060.2s 速度=0.05局/秒 预估剩余=480.7分钟
[200/1500] 累计局面数=24000 耗时=4108.5s 速度=0.05局/秒 预估剩余=445.1分钟
[300/1500] 累计局面数=36000 耗时=6120.2s 速度=0.05局/秒 预估剩余=408.0分钟
[400/1500] 累计局面数=48000 耗时=8135.1s 速度=0.05局/秒 预估剩余=372.9分钟
[500/1500] 累计局面数=60000 耗时=10173.2s 速度=0.05局/秒 预估剩余=339.1分钟
[600/1500] 累计局面数=72000 耗时=12192.5s 速度=0.05局/秒 预估剩余=304.8分钟
[700/1500] 累计局面数=84000 耗时=14207.9s 速度=0.05局/秒 预估剩余=270.6分钟
[800/1500] 累计局面数=96000 耗时=16255.0s 速度=0.05局/秒 预估剩余=237.1分钟
[900/1500] 累计局面数=108000 耗时=18269.2s 速度=0.05局/秒 预估剩余=203.0分钟
[1000/1500] 累计局面数=120000 耗时=20282.9s 速度=0.05局/秒 预估剩余=169.0分钟
[1100/1500] 累计局面数=132000 耗时=22319.3s 速度=0.05局/秒 预估剩余=135.3分钟
[1200/1500] 累计局面数=144000 耗时=24340.3s 速度=0.05局/秒 预估剩余=101.4分钟
[1300/1500] 累计局面数=156000 耗时=26357.4s 速度=0.05局/秒 预估剩余=67.6分钟
[1400/1500] 累计局面数=168000 耗时=28377.2s 速度=0.05局/秒 预估剩余=33.8分钟
[1500/1500] 累计局面数=180000 耗时=30361.6s 速度=0.05局/秒 预估剩余=0.0分钟
=== 数据整理中，正在保存文件 ===

✅ 生成完成！
总对局数：1500 局，总局面数：180000 个
总耗时：506.1 分钟
文件已保存至：/kaggle/working

#cell4 数据校验（可选，验证生成的文件是否正常）
# 加载生成的文件检查格式
data = np.load(GenConfig.out_path, allow_pickle=True)
print("=== 数据格式校验 ===")
print(f"features 形状: {data['features'].shape}, 类型: {data['features'].dtype}")
print(f"boards 形状: {data['boards'].shape}, 类型: {data['boards'].dtype}")
print(f"z_values 形状: {data['z_values'].shape}, 类型: {data['z_values'].dtype}")
print(f"move_scores 长度: {len(data['move_scores'])}, 单样本示例长度: {len(data['move_scores'][0])}")
print(f"moves_order 长度: {len(data['moves_order'])}, 单样本示例长度: {len(data['moves_order'][0])}")